# APK 4720 / APK 6725

# Assignment # 7

Please submit your assignment as a Jupyter  notebook (.ipynb file). Start a new Jupyter notebook and name it "YourName_Assignment_7"

Replace *YourName* with your first name and last name

>BEFORE STARTING THE ASSIGNMENT, MAKE SURE THAT **PYTORCH** IS INSTALLED IN YOUR ENVIRONMENT. 

Visit www.pytorch.org for instructions on how to install this package in your system.

## Background


For this assignment, we will use the *UCI HAR dataset* discussed in class. For more information about the data, please refer to the file 
> `05.02c CNN application.ipynb`

Briefly, this dataset contains data recorded from 30 volunteers performing six different activities, including 
1. Walking
2. Walking Upstairs
3. Walking Downstairs
4. Sitting
5. Standing
6. Laying

Participants wore a smartphone on their waist and data from the device's accelerometer and gyroscope was recorded. Data were collected at 50 samples/second, and then split into segments with a duration of 2.58s resulting in a total of 128 samples per recording. 

For each activity, there are a total of 9 channels, including:
- total acceleration: x, y, z
- body acceleration: x, y, z
- body gyroscope: x, y, z


The dataset is split into training and test, there are a total of 7,352 records for training and 2,947 records for testing. There are many records per subject (hundreds), and data from a subject was used in the training or testing sets only.

So, for each record, there are 9 signals, each of 128 samples long. 

Additional information can be found in the project website: https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones

## Part A (3 points)

In this section, you will design and train a convolutional neural network to classify the different activities present in the database. 

To start, download the file containing the data available in GitHub `Assingment_7_partA.npz`. This file contains the training and testing data for the first part of the experiment. 

Load the data into your notebook

In [8]:
import numpy as np
data = np.load('Assingment_7_partA.npz')

Replace `Assingment_7_partA.npz` with the path to your data in your computer. 

Note: In windows, you probably need to modify the path to `r"Assingment_7_partA.npz"`

Once you load the file, verify that the data is available, the run the following code to generate a plot of the 9 signals corresponding to one record from the data.

In [ ]:
import matplotlib.pyplot as plt

trainX = data['trainX']
trainy = data['trainy']
testX = data['testX']
testy = data['testy']

fig, ax = plt.subplots(3, 3, figsize=(15, 12), sharex = True)
signal_names = ['body_acc_x', 'body_acc_y', 'body_acc_z', 'body_gyro_x', 'body_gyro_y', 'body_gyro_z', 'total_acc_x', 'total_acc_y', 'total_acc_z']
for i, signal in enumerate(signal_names):
    sample_data = trainX[0]
    ax[i // 3, i % 3].plot(sample_data[:,i])
    ax[i // 3, i % 3].set_title(f'Sample {signal} Data')
plt.show()

**YOU HAVE TO ENSURE THAT THE PREVIOUS STEP WORKS. YOU CANNOT CONTINUE UNTIL YOU SEE A PLOT WITH THE DATA.**

If you don't see the figure with 9 panels and signals plotted on them, then there is a problem with the data. You need to fix the problem before continuing!

## Receptive fields
In `05.02c CNN application.ipynb` we built a simple CNN to classify activity based on the information in the database. We observed test accuracies close to 90%, indicating that the network did a good job at detecting the different classes from the information provided. 

One major question when creating the network was defining the size of the kernels that process the data. The kernel size is known as *receptive field* and can affect what kind of information the kernel extracts from the signal. Small kernels (i.e. size 3 to 5) focus on local changes in the signal, while larger kernels can focus on more global changes that occur in the signal. Clearly, picking the correct kernel size depends on the specific problem and can have significant effects in the results. 

To prevent this issue altogether, we can define a CNN that process the data with receptive fields of different sizes in parallel. In this way, the network itself will take care of selecting the optimal size of the receptive field through optimization. This kind of networks are called *Multi-Head* CNN, because they contain multiple heads that process the data in parallel. 

Let us create one such network and train it with the available data. 

In [ ]:
import torch
import torch.nn as nn
PYTORCH_ENABLE_MPS_FALLBACK=1 #this is needed if you have a mac, it allows you to use the GPU for training

#define a generic head, created by a convolutional block, followed by a nonlinearity, dropout, and max pooling
class ConvBranch(nn.Module):
    def __init__(self, in_channels, kernel_size, dropout=0.5):
        super().__init__()
        self.block = nn.Sequential(
            #layer 1
            nn.Conv1d(in_channels=in_channels, out_channels=32, kernel_size=kernel_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(kernel_size=2),

            #layer 2
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=kernel_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(kernel_size=2),
        )

    def forward(self, x):
        return self.block(x)

#define the full model, three heads with different kernel sizes, followed by a fully connected layer
class MultiReceptiveFieldCNN(nn.Module):
    def __init__(self, n_features, n_timesteps, n_outputs):
        super().__init__()

        # Same input, different temporal receptive fields
        self.branch_k3 = ConvBranch(in_channels=n_features, kernel_size=3) #receptive field of size 3 timesteps
        self.branch_k7 = ConvBranch(in_channels=n_features, kernel_size=7) #receptive field of size 7 timesteps
        self.branch_k17 = ConvBranch(in_channels=n_features, kernel_size=15) #receptive field of size 15 timesteps

        # Infer flattened output size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, n_features, n_timesteps)

            out1 = self.branch_k3(dummy).reshape(1, -1)
            out2 = self.branch_k7(dummy).reshape(1, -1)
            out3 = self.branch_k17(dummy).reshape(1, -1)

            merged_dim = out1.shape[1] + out2.shape[1] + out3.shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(merged_dim, 100),
            nn.ReLU(),
            nn.Linear(100, n_outputs),
        )

    def forward(self, x):
        #notice that each branch processes the same input, but with a different kernel size, allowing the model to learn features at different temporal scales
        b1 = self.branch_k3(x).reshape(x.size(0), -1)
        b2 = self.branch_k7(x).reshape(x.size(0), -1)
        b3 = self.branch_k17(x).reshape(x.size(0), -1)

        merged = torch.cat([b1, b2, b3], dim=1)
        out = self.classifier(merged)
        return out

After creating the network, run the following cell to count the number of parameters in the model. 

In [ ]:
# Build the model using the data dimensions
n_features = trainX.shape[2]      # 9 channels
n_timesteps = trainX.shape[1]     # 128 samples
n_outputs = len(np.unique(trainy))  # number of classes

model = MultiReceptiveFieldCNN(n_features, n_timesteps, n_outputs)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Questions:
- How many unknown parameters are in the model? 
- What algorithm can we use to identify these parameters? 

Once we have the model, we can train it using our data:

In [21]:
#helper function to train the model for one epoch and return the average loss and accuracy for that epoch.
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    y_true, y_pred, y_prob = [], [], []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)
        probs = torch.softmax(logits, dim=1)

        running_loss += loss.item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)

        y_true.append(yb.cpu().numpy())
        y_pred.append(logits.argmax(dim=1).cpu().numpy())
        y_prob.append(probs.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_prob = np.concatenate(y_prob)

    return running_loss / total, correct / total, y_true, y_pred, y_prob

In [ ]:
#train the model using the training dataset, and evaluate it using the test dataset
from torch.utils.data import DataLoader, Dataset
from torch import optim

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')


#define the dataset that will provide data to the model as it is being trained

class HARDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(np.transpose(X, (0, 2, 1)), dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = HARDataset(trainX, trainy)
test_dataset = HARDataset(testX, testy)


train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

n_features = trainX.shape[2]      # 9 channels
n_timesteps = trainX.shape[1]     # 128 samples
n_outputs = len(np.unique(trainy))  # number of classes

model = MultiReceptiveFieldCNN(n_features, n_timesteps, n_outputs).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 20 #you can increase this if you want, but it will take longer to train
history = []

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, y_true, y_pred, y_prob = evaluate_model(model, test_loader, criterion, device)

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'test_loss': test_loss,
        'test_acc': test_acc
    })

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
    )


In [ ]:
#after training, we can plot the training and test accuracy and loss curves to see how the model performed over time
import pandas as pd
history_df = pd.DataFrame(history)
plt.figure(figsize=(7, 4))
plt.plot(history_df['epoch'], history_df['train_acc'], marker='o', label='Train accuracy')
plt.plot(history_df['epoch'], history_df['test_acc'], marker='o', label='Test accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training history')
plt.legend()
plt.show()

In [ ]:
#and confusion matrix
class_names = ['Walking', 'Walking Upstairs', 'Walking Downstairs', 'Sitting', 'Standing', 'Laying']
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
cm_df

## Questions:
1. Go back to the notebook `05.02c CNN application.ipynb` and identify the results obtained with the network that we used in that notebook. How do those results compare with the ones you obtained using the multi-head CNN? 
2. The simple network analyzed in `05.02c CNN application.ipynb` has 199810 trainable parameters. With this information in mind, compare again the results obtained with both networks. 


### Extra activities (no value)
- Cut one of the heads in the network (you decide which) and re-train the model again. Do you see any difference?
- Remove the deepest layer in each head and re-train your model. Do you see any difference?


## Part B (2 points)

The CNN model identifies the best features for classification automatically. Another approach is to define a set of features and use these features as input to a classification model such as random forest or logistic regression. This approach is known as *feature engineering,* the features must be selected using domain knowledge, so they are often easier to interpret.   

In this section, you will train a model using features derived from the signals in the dataset. 

The authors of the study provided a list of 561 features extracted from the different signals in the dataset. The data is provided in a zip file `Assignment_7_partB.zip` available in GitHub. Unzip the file and load the training and test data.

In [51]:
features = pd.read_csv('Assignment_7_partB/features.txt', header=None, delim_whitespace=True, index_col=0, names=['feature_name'])
X_train = pd.read_csv('Assignment_7_partB/X_train.txt', header=None, index_col=None, delim_whitespace=True)
y_train = pd.read_csv('Assignment_7_partB/y_train.txt', header=None, index_col=None, delim_whitespace=True)
X_test = pd.read_csv('Assignment_7_partB/X_test.txt', header=None, index_col=None, delim_whitespace=True)
y_test = pd.read_csv('Assignment_7_partB/y_test.txt', header=None, index_col=None, delim_whitespace=True)

### Part B.1 (1 point)

Train and evaluate a random forest model, use grid search with cross-validation to identify the best model parameters. Uset the following script to define the model and grid.
```Python
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestClassifier
    pipeline = Pipeline([
    ('model', RandomForestClassifier(class_weight='balanced',random_state=42,n_jobs=-1)) #n_jobs=-1 to use all the available cores for training the random forest model
    ])

    params_grid_RandomForest = {
        #parameters for the random forest model
        'model__n_estimators': [50, 200, 500],
        'model__max_depth': [None, 10, 20, 30],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
    }
```

After training the model, compute the test set accuracy and confusion matrix. 

## Questions:
1. What is the model accuracy?
2. How does the model's confusion matrix compares to the one estimated with the CNN

### Part B.2 (1 point)

Train and evaluate a logistic regression model. This model will take a long time to train, so we will only use default parameters. Use the following script to define the model 
```Python
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    pipeline = Pipeline([
        ('scaler', StandardScaler(),),
        ('model', LogisticRegression(penalty='l1',solver='saga',max_iter=5000,random_state=42,n_jobs=-1, class_weight='balanced')
    )
    ])
```

After training the model, compute the test set accuracy and confusion matrix. 

## Questions:
1. What is the parameter `penalty='l1'` doing? How does this parameter influence the model? 
2. What is the model accuracy?
3. How does the model's confusion matrix compares to the one estimated with the CNN
4. Considering the results obtained with the CNN, Random Forest, and Logistic regression models, which model do you think is the best? 

## Extra (no value)

Compute the 10 most important features of the Random fores and Logistic Regression models. Are these the same? What do these features tell you about the models?